.

In [1]:
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display_html

# Explicitly re-enable text/html processing in the active shell formatter
ip = InteractiveShell.instance()
display_formatter = ip.display_formatter
if display_formatter is not None:
    display_formatter.formatters["application/javascript"].enabled = True
    display_formatter.formatters["image/jpeg"].enabled = True
    display_formatter.formatters["image/png"].enabled = True
    display_formatter.formatters["image/svg+xml"].enabled = True
    display_formatter.formatters["text/html"].enabled = True
    display_formatter.formatters["text/markdown"].enabled = True

# Define the table data

"""
Initial Investment         = $1,000,000.00
Total S&P 500 Benchmark    = $1,444,620.22 = +444,620.22$ = +44.46% (total) =  +8.53% (annualized)
Total Portfolio            = $1,347,278.74 = +347,278.74$ = +34.73% (total) =  +6.90% (annualized)
Total Portfolio Rebalanced = $1,347,035.37 = +347,035.37$ = +34.70% (total) =  +6.90% (annualized)
"""
# 1. Create a sample DataFrame with None/NaN values
rows = ["Initial Investment", "S&P 500 Benchmark", "Portfolio", "Portfolio Rebalanced"]
columns = ["Value", "Growth", "Total", "Annualized"]
data = {
    columns[0]: [1_000_000.00, 1, 222, 32222],
    columns[1]: [None, -0.042, 15, 125],
    columns[2]: [None, 3.912, 13, 12],
    columns[3]: [None, 3.912, 11, 12],
}

df = pd.DataFrame(data, index=rows)

# 2. Define your column formatting
column_formats = {
    columns[0]: "${:,.2f}",  # Value
    columns[1]: "{:+,.2f}$",  # Growth
    columns[2]: "{:+,.2f}%",  # Total %
    columns[3]: "{:+,.2f}%",  # Annualized %
}

styled_df = df.copy()
print("Output of print:")
print(styled_df)

for col, fmt in column_formats.items():
    styled_df[col] = styled_df[col].apply(lambda x, fmt=fmt: "" if pd.isna(x) else fmt.format(x))

styled = (
    styled_df.style.map_index(lambda v: "font-weight: bold;", axis=0).format(na_rep="")  # Displays missing values as completely empty
)
# Display in Jupyter Notebook or export to HTML
print("Output of display_html:")
with pd.option_context("display.notebook_repr_html", True):
    display_html(styled.to_html(), raw=True)

# Use tabulate formats for pretty CLI output (options like 'github', 'grid', or 'fancy_grid')
print("Output of print(to_markdown):")
print(styled_df.to_markdown(tablefmt="psql", colalign=["left", "right", "right", "right", "right"]))

Output of print:
                          Value   Growth   Total  Annualized
Initial Investment    1000000.0      NaN     NaN         NaN
S&P 500 Benchmark           1.0   -0.042   3.912       3.912
Portfolio                 222.0   15.000  13.000      11.000
Portfolio Rebalanced    32222.0  125.000  12.000      12.000
Output of display_html:


,Value,Growth,Total,Annualized
Initial Investment,"$1,000,000.00",,,
S&P 500 Benchmark,$1.00,-0.04$,+3.91%,+3.91%
Portfolio,$222.00,+15.00$,+13.00%,+11.00%
Portfolio Rebalanced,"$32,222.00",+125.00$,+12.00%,+12.00%


Output of print(to_markdown):
+----------------------+---------------+----------+---------+--------------+
|                      |         Value |   Growth |   Total |   Annualized |
|----------------------+---------------+----------+---------+--------------|
| Initial Investment   | $1,000,000.00 |          |         |              |
| S&P 500 Benchmark    |         $1.00 |   -0.04$ |  +3.91% |       +3.91% |
| Portfolio            |       $222.00 |  +15.00$ | +13.00% |      +11.00% |
| Portfolio Rebalanced |    $32,222.00 | +125.00$ | +12.00% |      +12.00% |
+----------------------+---------------+----------+---------+--------------+


In [3]:
import pandas as pd
from tabulate import tabulate

# Define header pairs representing Level 0 and Level 1
headers = [
    ("Holding", "Ticker"),
    ("Holding", "Book"),
    ("Portfolio", "Shares"),
    ("Portfolio", "Percent"),
    ("Portfolio", "Value"),
    ("Portfolio", "Annualized"),
    ("Rebalanced", "Shares"),
    ("Rebalanced", "Percent"),
    ("Rebalanced", "Value"),
    ("Rebalanced", "Annualized"),
]

# Provide 2D data that matches the column count
data = [["VOO", 1, 10, 20, 30, 1, 1, 1, 1, 1], ["SPY", 200, 4000, 500, 600, 20, 3, 4000, 5898, 6.879876978]]

# Build the DataFrame
df = pd.DataFrame(data, columns=pd.MultiIndex.from_tuples(headers))

styled_df = df.copy()
# 2. Define your column formatting
column_formats = {
    headers[0]: "{}",  # Ticker
    headers[1]: "${:,}",  # Book
    headers[2]: "{:,.2f}",  # Shares
    headers[3]: "{:,.2f}%",  # Percent
    headers[4]: "${:,.2f}",  # Value
    headers[5]: "{:+,.2f}%",  # Annualized
    headers[6]: "{:,.2f}",  # Shares
    headers[7]: "{:,.2f}%",  # Percent
    headers[8]: "${:,.2f}",  # Value
    headers[9]: "{:+,.2f}%",  # Annualized
}
for col, fmt in column_formats.items():
    styled_df[col] = styled_df[col].apply(lambda x, fmt=fmt: "" if pd.isna(x) else fmt.format(x))
df = styled_df

styled = (
    df.style.map_index(lambda v: "font-weight: bold;", axis=0).format(na_rep="")  # Displays missing values as completely empty
)
# 2. Add a heavy right border to the last column of "Group A"
styled = styled.set_properties(subset=[("Holding", "Book"), ("Portfolio", "Annualized")], **{"border-right": "2px solid black"})

# 2. Target only the level 0 column headers
styled = styled.set_table_styles([{"selector": "th.col_heading.level0", "props": [("text-align", "left")]}])
# Display in Jupyter Notebook or export to HTML
print("Output of display_html:")
with pd.option_context("display.notebook_repr_html", True):
    display_html(styled.to_html(), raw=True)

# Extract header rows (exclude the index column)
top_headers = df.columns.get_level_values(0)
is_duplicate = pd.Series(top_headers).duplicated()
row_0_headers = ["" if dup else label for label, dup in zip(top_headers, is_duplicate, strict=True)]
row_1_headers = [col[1] for col in df.columns]
# Format and print
print("Output of tabulate:")
print(tabulate([row_1_headers] + df.values.tolist(), headers=row_0_headers, tablefmt="simple", stralign="right"))

Output of display_html:


Holding 
 Portfolio 
 Rebalanced 
 
 
   
 Ticker 
 Book 
 Shares 
 Percent 
 Value 
 Annualized 
 Shares 
 Percent 
 Value 
 Annualized 
 
 
 
 
 0 
 VOO 
 $1 
 10.00 
 20.00% 
 $30.00 
 +1.00% 
 1.00 
 1.00% 
 $1.00 
 +1.00% 
 
 
 1 
 SPY 
 $200 
 4,000.00 
 500.00% 
 $600.00 
 +20.00% 
 3.00 
 4,000.00% 
 $5,898.00 
 +6.88%

Output of tabulate:
  Holding          Portfolio                                  Rebalanced
---------  ----  -----------  -------  -------  ----------  ------------  ---------  ---------  ----------
   Ticker  Book       Shares  Percent    Value  Annualized        Shares    Percent      Value  Annualized
      VOO    $1        10.00   20.00%   $30.00      +1.00%          1.00      1.00%      $1.00      +1.00%
      SPY  $200     4,000.00  500.00%  $600.00     +20.00%          3.00  4,000.00%  $5,898.00      +6.88%
